# 01 · Recolección de datos

**Objetivo.** Descargar de forma reproducible las cuatro fuentes del proyecto (precios de yfinance, hechos contables de SEC EDGAR, tablas por industria de Damodaran y el `info` de yfinance), guardarlas como instantáneas con fecha y documentar qué se obtuvo.

**Punto del enunciado que cubre.** Punto 5, Recolección de datos (inventario de fuentes, modelo de entidades, diccionario y proceso de recolección).

**Hipótesis que aborda.** Provee los datos de todas: HT0–HT4 (precios), HF1–HF4 (precios más fundamentales), HF5 (triangulación con `info` y Damodaran) y HP0–HP3 (precios y ETF sectoriales).

**Entradas.** Internet: yfinance, `sec.gov`/`data.sec.gov` y `pages.stern.nyu.edu`. La variable de entorno `SEC_USER_AGENT` identifica al usuario ante la SEC (su valor nunca se imprime ni se guarda).

**Salidas (instantáneas en `data/snapshots/`).** `precios_semanales`, `precios_mensuales`, `edgar_hechos`, `ratios_anuales`, `damodaran_<clave>` e `info_yf`. Los notebooks 02 a 04 las leen con `datos.cargar_snapshot`.

**Diccionario de datos.** Ver [`docs/entrega_1/05_diccionario_datos.md`](../docs/entrega_1/05_diccionario_datos.md).

In [1]:
import io, os
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import requests, yfinance as yf
from asistente import config, datos, edgar, fundamentales as fu, damodaran as dm
sns.set_theme(style="whitegrid"); pd.set_option("display.float_format", "{:.4f}".format)
print("Directorio de instantáneas:", config.DIR_SNAPSHOTS.relative_to(config.RAIZ))
print("SEC_USER_AGENT definido:", bool(os.environ.get("SEC_USER_AGENT")))

Directorio de instantáneas: data/snapshots
SEC_USER_AGENT definido: True


## 1. Precios semanales y mensuales (yfinance)

Se descargan los 30 tickers del universo (tres industrias de diez empresas), el benchmark SPY y los tres ETF sectoriales, con precios ajustados por dividendos y particiones (`auto_adjust=True`). La descarga semanal alimenta el eje técnico y el probabilístico; la mensual se usa como chequeo de consistencia y para el momentum `mom_12_1`.

In [2]:
todos = config.tickers() + [config.BENCHMARK] + list(config.ETF_SECTOR.values())
# yfinance ancla las barras semanales al día de la semana de la fecha de inicio: se arranca en el primer lunes
inicio = (pd.Timestamp(config.INICIO) + pd.offsets.Week(weekday=0)).strftime("%Y-%m-%d")     if pd.Timestamp(config.INICIO).dayofweek != 0 else config.INICIO
print("config.INICIO:", config.INICIO, "| inicio semanal usado (lunes):", inicio)
sem, falt_s = datos.descargar_precios(todos, inicio, intervalo="1wk")
men, falt_m = datos.descargar_precios(todos, config.INICIO, intervalo="1mo")
print("tickers solicitados:", len(todos))
print("faltantes semanal:", falt_s, "| mensual:", falt_m)
resumen_px = sem.groupby("ticker").fecha.agg(["min", "max", "count"])
print(resumen_px.sort_values("min").to_string())
print("Día de la semana de las barras semanales (0 = lunes):", sem.fecha.dt.dayofweek.value_counts().to_dict())
print("Fechas semanales distintas en todo el panel:", sem.fecha.nunique())
print("Día del mes de las barras mensuales:", men.fecha.dt.day.value_counts().to_dict())

config.INICIO: 2009-01-01 | inicio semanal usado (lunes): 2009-01-05


tickers solicitados: 34
faltantes semanal: [] | mensual: []
              min        max  count
ticker                             
AAPL   2009-01-05 2026-10-12    928
XLF    2009-01-05 2026-10-12    928
WMT    2009-01-05 2026-10-12    928
WFC    2009-01-05 2026-10-12    928
USB    2009-01-05 2026-10-12    928
TFC    2009-01-05 2026-10-12    928
SPY    2009-01-05 2026-10-12    928
SCHW   2009-01-05 2026-10-12    928
PNC    2009-01-05 2026-10-12    928
PG     2009-01-05 2026-10-12    928
PEP    2009-01-05 2026-10-12    928
ORCL   2009-01-05 2026-10-12    928
NVDA   2009-01-05 2026-10-12    928
MSFT   2009-01-05 2026-10-12    928
MS     2009-01-05 2026-10-12    928
XLK    2009-01-05 2026-10-12    928
MDLZ   2009-01-05 2026-10-12    928
CSCO   2009-01-05 2026-10-12    928
ADBE   2009-01-05 2026-10-12    928
BAC    2009-01-05 2026-10-12    928
C      2009-01-05 2026-10-12    928
CL     2009-01-05 2026-10-12    928
COST   2009-01-05 2026-10-12    928
CRM    2009-01-05 2026-10-12    928
KO  

In [3]:
datos.guardar_snapshot(sem, "precios_semanales"); datos.guardar_snapshot(men, "precios_mensuales")
corta = resumen_px[resumen_px["min"] > pd.Timestamp(config.INICIO) + pd.Timedelta(days=31)]
print("Tickers cuya historia empieza después de enero de 2009:")
print(corta.to_string())

Tickers cuya historia empieza después de enero de 2009:
              min        max  count
ticker                             
KHC    2015-07-06 2026-10-12    589
META   2012-05-14 2026-10-12    753


**Lectura.** Se pidieron 34 símbolos y la lista de faltantes quedó vacía en las dos frecuencias. El panel semanal tiene 928 fechas distintas y todas caen en lunes (31038 filas en total); las mensuales caen todas el día 1 del mes. La historia es más corta para dos empresas: META empieza el 2012-05-14 (753 semanas) porque salió a bolsa en 2012, y KHC el 2015-07-06 (589 semanas) porque nació de la fusión de Kraft y Heinz en 2015. Esa historia corta es esperada y no un error de descarga; los análisis que necesiten ventanas largas (por ejemplo, los percentiles expansivos de 52 semanas) las usarán con menos observaciones.

**Justificación.** El primer intento de descarga empezó el 2009-01-01 (un jueves) y yfinance ancló las barras semanales a ese día para casi todos los tickers, mientras que META y KHC quedaron en lunes: el panel tenía muchas más fechas distintas que las del calendario común, y un `pivot` ancho habría dejado huecos. Por eso se arranca en el primer lunes (2009-01-05): todas las series comparten calendario y las uniones por fecha (con ETF y SPY) son exactas. La instantánea mensual no tiene el problema, porque yfinance ancla esas barras al día 1.

## 2. Hechos contables (SEC EDGAR)

Se usa la API `companyfacts` (XBRL, taxonomía `us-gaap`). Primero se mapea cada ticker a su CIK con `company_tickers.json`; después se descarga el JSON de cada empresa, se convierte a tabla y se filtra a los diez conceptos de `edgar.CONCEPTOS`. La SEC exige un `User-Agent` que identifique al solicitante y limita a 10 solicitudes por segundo; `edgar.descargar_json` ya lo lee de `SEC_USER_AGENT`.

In [4]:
ciks = edgar.mapa_ticker_cik(edgar.descargar_json(edgar.URL_TICKERS))
sin_cik = [t for t in config.tickers() if t not in ciks]
print("Tickers del universo sin CIK en company_tickers.json:", sin_cik)

Tickers del universo sin CIK en company_tickers.json: []


In [5]:
nombres = [n for alt, _ in edgar.CONCEPTOS.values() for n in alt]
hechos, ratios, errores = [], [], {}
for t in config.tickers():
    if t not in ciks:
        errores[t] = "sin CIK"; continue
    try:
        h = edgar.hechos_a_df(edgar.descargar_json(edgar.URL_FACTS.format(cik=ciks[t])))
        hechos.append(h[h.concepto.isin(nombres)].assign(ticker=t))
        ratios.append(fu.ratios_anuales(fu.tabla_anual(h)).assign(ticker=t))
    except Exception as e:
        errores[t] = f"{type(e).__name__}: {str(e)[:80]}"
edgar_hechos = pd.concat(hechos, ignore_index=True); ratios_anuales = pd.concat(ratios, ignore_index=True)
datos.guardar_snapshot(edgar_hechos, "edgar_hechos"); datos.guardar_snapshot(ratios_anuales, "ratios_anuales")
print("errores:", errores or "ninguno")
print(ratios_anuales.groupby("ticker").end.agg(["min", "max", "count"]).to_string())

errores: ninguno
              min        max  count
ticker                             
AAPL   2006-09-30 2025-09-27     20
ADBE   2006-12-01 2025-11-28     20
BAC    2006-12-31 2025-12-31     20
C      2006-12-31 2025-12-31     20
CL     2006-12-31 2025-12-31     20
COST   2007-09-02 2026-08-30     20
CRM    2007-01-31 2026-01-31     20
CSCO   2007-07-28 2026-07-25     20
GIS    2007-05-27 2026-05-31     20
GOOGL  2012-12-31 2025-12-31     14
GS     2007-11-30 2025-12-31     21
INTC   2006-12-30 2025-12-27     20
JPM    2007-12-31 2025-12-31     19
KHC    2013-12-29 2025-12-27     19
KMB    2006-12-31 2025-12-31     20
KO     2006-12-31 2025-12-31     20
MDLZ   2006-12-31 2025-12-31     20
META   2009-12-31 2025-12-31     17
MS     2006-11-30 2025-12-31     21
MSFT   2007-06-30 2026-06-30     20
NVDA   2007-01-28 2026-01-25     20
ORCL   2007-05-31 2026-05-31     20
PEP    2006-12-30 2025-12-27     20
PG     2007-06-30 2026-06-30     20
PNC    2006-12-31 2025-12-31     21
SCHW   2006

In [6]:
# Cobertura por concepto: cuántas empresas-año tienen valor en la tabla anual
cols = list(edgar.CONCEPTOS)
cobertura = ratios_anuales[cols].notna().mean().rename("fraccion_con_valor").to_frame()
cobertura["empresas_con_algun_valor"] = ratios_anuales.groupby("ticker")[cols].apply(lambda d: d.notna().any()).sum()
print(cobertura.to_string())

                    fraccion_con_valor  empresas_con_algun_valor
ingresos                        0.7635                        27
utilidad_bruta                  0.4037                        14
utilidad_operativa              0.6149                        20
utilidad_neta                   0.9392                        30
patrimonio                      0.9848                        30
deuda_lp                        0.6503                        29
caja                            0.7855                        28
cfo                             0.8480                        30
capex                           0.6064                        22
acciones                        0.9155                        30


**Lectura.** Los 30 tickers tienen CIK y no hubo errores de descarga. En la cobertura por concepto, patrimonio (0.98 de las empresas-año), utilidad neta (0.94) y acciones (0.92) son los más completos; utilidad bruta es el más escaso: solo 14 de 30 empresas lo reportan (0.40 de las empresas-año), y capex lo reporta 22 de 30. Los bancos no publican `GrossProfit` y varias empresas usan etiquetas XBRL distintas; esos vacíos no son ceros (ver el diccionario de datos). GOOGL tiene 14 años de ratios (el primero cierra el 2012-12-31) y META 17 (desde 2009-12-31), frente a 19 a 21 de la mayoría.

**Justificación.** Se conservan todas las filas con vacíos en lugar de imputarlas: un margen bruto de un banco no existe, y rellenarlo con 0 haría pasar una ausencia por una cifra. Los ratios que dependen de un concepto ausente quedan como `NaN`, y el notebook de calidad (02) contará cuántos casos hay por ratio. La unicidad se revisa más abajo sobre las llaves completas de cada hecho.

## 3. Tablas por industria (Damodaran) e `info` de yfinance

Damodaran publica benchmarks por industria (retorno sobre capital, EV/EBITDA, P/E, márgenes) en archivos Excel de acceso libre. Se descarga cada URL de `damodaran.URLS`; si alguna falla se registra el código HTTP y se continúa. El `info` de yfinance se baja ticker por ticker y sirve para triangular los ratios calculados desde EDGAR (HF5, lección E1 del examen 1: validar lo que entrega el proveedor).

In [7]:
estado_dm = {}
for clave, url in dm.URLS.items():
    try:
        r = requests.get(url, timeout=60, headers={"User-Agent": "Mozilla/5.0"})
        estado_dm[clave] = r.status_code
        if r.ok:
            datos.guardar_snapshot(dm.leer_tabla(io.BytesIO(r.content)), f"damodaran_{clave}")
    except Exception as e:
        estado_dm[clave] = f"{type(e).__name__}: {str(e)[:80]}"
print("Estado de descarga por clave:", estado_dm)

Estado de descarga por clave: {'roc': 200, 'ev_ebitda': 200, 'pe': 200, 'margenes': 200}


In [8]:
campos = ["sector", "industry", "trailingPE", "priceToBook", "returnOnEquity", "marketCap"]
filas_info, fallos_info = [], {}
for t in config.tickers():
    try:
        inf = yf.Ticker(t).info
        filas_info.append({"ticker": t, **{c: inf.get(c) for c in campos}})
    except Exception as e:
        fallos_info[t] = f"{type(e).__name__}: {str(e)[:80]}"
info = pd.DataFrame(filas_info)
datos.guardar_snapshot(info, "info_yf")
print("fallos en info:", fallos_info or "ninguno")
print("Valores vacíos por campo:"); print(info.isna().sum().to_string())
print(info.head(6).to_string())

fallos en info: ninguno
Valores vacíos por campo:
ticker            0
sector            0
industry          0
trailingPE        3
priceToBook       0
returnOnEquity    0
marketCap         0
  ticker                  sector                        industry  trailingPE  priceToBook  returnOnEquity      marketCap
0   AAPL              Technology            Consumer Electronics     39.0916      46.3682          1.4875  4980555644928
1   MSFT              Technology       Software - Infrastructure     30.2758       9.1236          0.3404  4035413016576
2  GOOGL  Communication Services  Internet Content & Information     17.6978       6.9510          0.4868  4326706380800
3   META  Communication Services  Internet Content & Information     27.9919       7.4376          0.2985  1942473408512
4   NVDA              Technology                  Semiconductors     28.8900      24.0979          1.1721  5518072479744
5   ORCL              Technology       Software - Infrastructure     22.4451       7

In [9]:
# Vista previa de una tabla de Damodaran (industrias que usa MAPA_INDUSTRIAS)
for clave in dm.URLS:
    try:
        t = datos.cargar_snapshot(f"damodaran_{clave}")
    except FileNotFoundError:
        print(clave, ": no disponible"); continue
    print(clave, t.shape, list(t.columns)[:6])
t = datos.cargar_snapshot("damodaran_pe") if (config.DIR_SNAPSHOTS / "damodaran_pe.parquet").exists() else None
if t is not None:
    print(dm.filas_industria(t, "tecnologia").iloc[:, :5].to_string())

roc (96, 9) ['industry_name', 'number_of_firms', 'unadjusted_pre_tax_roic', 'lease_adjusted_pre_tax_roic', 'lease_r_d_adjusted_pre_tax_roic', 'unadjusted_after_tax_roic']
ev_ebitda (96, 10) ['industry_name', 'number_of_firms', 'ev_ebitdar_d', 'ev_ebitda', 'ev_ebit', 'ev_ebit_1_t']
pe (96, 10) ['industry_name', 'number_of_firms', 'of_money_losing_firms_trailing', 'current_pe', 'trailing_pe', 'forward_pe']
margenes (96, 19) ['industry_name', 'number_of_firms', 'gross_margin', 'net_margin', 'pre_tax_pre_stock_compensation_operating_margin', 'pre_tax_unadjusted_operating_margin']
                     industry_name  number_of_firms  of_money_losing_firms_trailing  current_pe  trailing_pe
0            Computers/Peripherals               36                          0.6111     80.9453      81.1320
1                    Semiconductor               66                          0.6212     70.1348     100.1844
2              Software (Internet)               29                          0.7241    162

**Lectura.** Las cuatro descargas de Damodaran devolvieron el código HTTP 200 y cada tabla tiene 96 filas (industrias). La cobertura de `info` es completa salvo `trailingPE`, que tiene 3 vacíos de 30 empresas (no se imputan; el P/E de esas empresas se calcula desde EDGAR cuando hay utilidad positiva). Las industrias que usa `damodaran.MAPA_INDUSTRIAS` para tecnología aparecen todas en la tabla de P/E, así que el cruce industria a industria no pierde filas.

**Justificación.** Los nombres de industria de yfinance (por ejemplo "Consumer Electronics") no coinciden con los de Damodaran, por eso el enlace no se hace por texto sino con el mapa explícito `MAPA_INDUSTRIAS` y el universo de `config`. La coherencia entre ambas clasificaciones se mide en el notebook de calidad (HF5).

## 4. Fuentes

### 4.1 Inventario

| Fuente | URL | Estructura | Granularidad | Cobertura | Acceso y términos | Importancia | Hipótesis |
|---|---|---|---|---|---|---|---|
| **yfinance (precios)** | https://finance.yahoo.com (paquete `yfinance`) | Serie de tiempo en panel (ticker × fecha) | Semanal y mensual; `close` ajustado y `volume` | 34 símbolos (30 acciones, SPY y 3 ETF) desde 2009 (META desde 2012) | Gratuito, sin llave. Uso personal y no comercial según los términos de Yahoo; no es una API oficial | **Crítica** | HT0–HT4, HP0–HP3, HF1–HF4 (precio del múltiplo) |
| **SEC EDGAR `companyfacts`** | https://data.sec.gov/api/xbrl/companyfacts/ | Hechos contables en formato largo (concepto × periodo × fecha `filed`) | Anual (10-K, año fiscal completo) con fecha de publicación; solo se usan los datos anuales | 30 empresas, datos anuales desde 2006 (GOOGL desde 2012) hasta el último 10-K | Dominio público; exige cabecera `User-Agent` identificada y máximo 10 solicitudes por segundo | **Crítica** | HF1–HF5 |
| **Damodaran Online** | https://pages.stern.nyu.edu/~adamodar/ | Tabla de referencia por industria (una fila por industria) | Anual (enero de cada año) | ~95 industrias de EE. UU. | Gratuito para uso académico; se cita la fuente | Complementaria | HF2, HF5 |
| **yfinance `info`** | https://finance.yahoo.com (`Ticker.info`) | Tabla de referencia (una fila por empresa) | Foto del día de descarga | 30 empresas | Igual que los precios; los campos cambian sin aviso | Complementaria | HF5 (triangulación) |

La importancia es crítica cuando sin la fuente no se puede responder ninguna hipótesis principal (★); es complementaria cuando solo sirve para validar o para dar contexto.

### 4.2 Modelo de entidades

```mermaid
erDiagram
    EMPRESA ||--o{ PRECIO_SEMANAL : "ticker"
    EMPRESA ||--o{ HECHO_CONTABLE : "CIK"
    EMPRESA }o--|| INDUSTRIA : "industria"
    EMPRESA }o--|| ETF : "sector"
    ETF ||--o{ PRECIO_SEMANAL : "ticker"
    EMPRESA {
        string ticker PK
        string cik
        string industria FK
        string etf FK
    }
    PRECIO_SEMANAL {
        string ticker FK
        date fecha
        float close
        float volume
    }
    HECHO_CONTABLE {
        string cik FK
        string concepto
        date end
        date filed
        float val
    }
    INDUSTRIA {
        string industria PK
        float roc
        float pe
        float ev_ebitda
    }
    ETF {
        string ticker PK
        string industria FK
    }
```

Los precios y los hechos contables se unen por empresa, y en el tiempo por la fecha `filed` (la fecha en que el 10-K fue público), no por el cierre del año fiscal: así se evita el sesgo de anticipación (*look-ahead*). La industria de cada empresa sale de `config.UNIVERSO`; las tablas de Damodaran se enlazan por `damodaran.MAPA_INDUSTRIAS`.

### 4.3 Volumen descargado

La tabla sale de los metadatos de cada instantánea y del tamaño de cada archivo.

In [10]:
import json
filas = []
for meta in sorted(config.DIR_SNAPSHOTS.glob("*.meta.json")):
    m = json.loads(meta.read_text()); pq = config.DIR_SNAPSHOTS / f"{m['nombre']}.parquet"
    df = pd.read_parquet(pq)
    filas.append({"instantanea": m["nombre"], "filas": m["filas"], "columnas": df.shape[1],
                  "kb": pq.stat().st_size / 1024, "fecha_descarga_utc": m["fecha_descarga"]})
volumen = pd.DataFrame(filas)
print(volumen.to_string(index=False))
print("Total de filas:", volumen.filas.sum(), "| tamaño total (MB):", round(volumen.kb.sum() / 1024, 2))

        instantanea  filas  columnas       kb        fecha_descarga_utc
damodaran_ev_ebitda     96        10  14.7773 2026-10-14T05:18:42+00:00
 damodaran_margenes     96        19  29.1855 2026-10-14T05:18:43+00:00
       damodaran_pe     96        10  15.5654 2026-10-14T05:18:43+00:00
      damodaran_roc     96         9  14.4043 2026-10-14T05:18:42+00:00
       edgar_hechos  70790         9 461.6240 2026-10-14T05:18:41+00:00
            info_yf     30         7   6.0352 2026-10-14T05:18:57+00:00
  precios_mensuales   7158         4 112.5449 2026-10-14T05:18:07+00:00
  precios_semanales  31038         4 483.9961 2026-10-14T05:18:07+00:00
     ratios_anuales    592        23  87.3975 2026-10-14T05:18:41+00:00
Total de filas: 109992 | tamaño total (MB): 1.2


In [11]:
# Chequeo de integridad antes de cerrar: ¿se pueden volver a cargar y tienen las llaves esperadas?
for nombre, llaves in [("precios_semanales", ["fecha", "ticker"]), ("precios_mensuales", ["fecha", "ticker"]),
                       ("edgar_hechos", ["ticker", "concepto", "unidad", "start", "end", "filed", "form", "fp"]), ("ratios_anuales", ["ticker", "end"])]:
    d = datos.cargar_snapshot(nombre)
    print(f"{nombre:18s} filas={len(d):>7d} llaves duplicadas={d.duplicated(llaves).sum()}")

precios_semanales  filas=  31038 llaves duplicadas=0
precios_mensuales  filas=   7158 llaves duplicadas=0
edgar_hechos       filas=  70790 llaves duplicadas=33
ratios_anuales     filas=    592 llaves duplicadas=0


**Lectura.** Las instantáneas suman 109992 filas y 1.2 MB, así que caben en el repositorio. Los precios semanales y mensuales no tienen llaves duplicadas (ticker y fecha), ni los ratios anuales (ticker y fecha de cierre). `edgar_hechos` sí tiene 33 filas repetidas sobre sus llaves completas de 70790 (una fracción de 0.05 %); es una cantidad menor y `edgar.serie_anual` se queda con un valor por periodo, por eso no se eliminan aquí.

**Síntesis**
- Se descargaron 34 símbolos con precios semanales (928 fechas, todas lunes) y mensuales, sin faltantes; META (2012) y KHC (2015) tienen historia corta, esperada.
- Los 30 tickers tienen hechos de EDGAR y ratios anuales; los vacíos (sobre todo utilidad bruta y capex) se dejan como ausentes y no como cero.
- Damodaran (4 tablas de 96 industrias) e `info` de yfinance (30 empresas) quedaron guardados para triangular en el notebook de calidad (HF5).
- Decisión que alimenta: todo el análisis posterior parte de estas instantáneas con fecha, de modo que los resultados son reproducibles aunque las fuentes cambien.